# PyLag Tutorial: Lateral advection and diffusion
**Author Jun Sasaki coded on 2024-03-27, updated on 2024-04-05**<br>
[Official tutorial](https://pylag.readthedocs.io/en/latest/examples/lateral_adv_diff_analytic.html) is modified.

In [ ]:
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from IPython.display import HTML

from pylag.mock import MockVelocityEddyDiffusivityDataReader
from pylag.processing.plot import create_figure, colourmap
from pylag.processing.utils import get_grid_bands

warnings.filterwarnings('ignore')
# Ensure inline plotting
%matplotlib inline

## Set up model

In [ ]:
xmin = -20.
ymin = -20.
xmax = 60.
ymax = 60.
n_x_points = 41
n_y_points = 41
x_grid = np.linspace(xmin, xmax, n_x_points, dtype=float)
y_grid = np.linspace(ymin, ymax, n_y_points, dtype=float)

# Grid edges for plotting
x_grid_edges = get_grid_bands(x_grid)
y_grid_edges = get_grid_bands(y_grid)

# Set up the time grid
t_min = 0.1
t_max = 25.0
time_step = 0.1
t_grid = time = np.arange(t_min, t_max + time_step, time_step)
n_times = len(t_grid)

## Compute the analytic solution

In [ ]:
data_reader = MockVelocityEddyDiffusivityDataReader()
C = np.empty((n_times, n_x_points, n_y_points), dtype=float)
for t_idx, t in enumerate(t_grid):
    for x_idx, x in enumerate(x_grid):
        for y_idx, y in enumerate(y_grid):
            C[t_idx, x_idx, y_idx] = data_reader.get_concentration_analytic(t, x, y)

## Animate the analytical concentration field

In [ ]:
font_size = 15
fig, ax = create_figure(figure_size=(16., 16.), font_size=font_size)
ax.set_xlim((x_grid_edges.min(), x_grid_edges.max()))
ax.set_ylim((y_grid_edges.min(), y_grid_edges.max()))
ax.set_xlabel('x', fontsize=font_size)
ax.set_ylabel('y', fontsize=font_size)
ax.set_aspect('equal')

# Set colour map and limits
cmap = colourmap('DYE')
vmin = 0.0
vmax = 10.0

# Set up the plot
pcolormesh= ax.pcolormesh(x_grid_edges, y_grid_edges, C[0,:,:], vmin=vmin, vmax=vmax, cmap=cmap)
pos = ax.get_position().get_points()
cax = fig.add_axes([pos[1,0]+0.02, pos[0,1], 0.02, pos[1,1] - pos[0,1]])
cbar = fig.colorbar(pcolormesh, cax=cax)
cbar.ax.tick_params(labelsize=font_size)
cbar.set_label('C (kg m$\mathrm{^{-3}}$)', fontsize=font_size)

def animate(i):
    pcolormesh.set_array(C[i,:,:].ravel())
    return (pcolormesh,)

# Prevent the basic figure from being plotted too
plt.close(fig)

anim = animation.FuncAnimation(fig, animate, frames=n_times, interval=40, blit=True)
anim.save("test01.gif")

HTML(anim.to_html5_video())

## Running the particle tracking model
The original tutorial has a bug of deactivated advection as reported [here](https://github.com/pmlmodelling/pylag/issues/144).
Advection can be considered by adding ocean currents as follows:
```Python
config.add_section("OCEAN_DATA")
config.set("OCEAN_DATA", "name", "dummy")
```
which is implemented in [velocity_aggregator.pyx](https://github.com/jsasaki-utokyo/pylag/blob/main/pylag/velocity_aggregator.pyx).

Velocities are set at Line 155-159:
```Python
# Ocean transport (three component)
if self._apply_ocean_velocity_term == True:
    data_reader.get_velocity(time, particle, ocean_velocity)
        for i in range(3):
            velocity[i] += ocean_velocity[i]
```
Velocities are defined in `MockVelocityEddyDiffusivityDataReader` in [mock.pyx](https://github.com/pmlmodelling/pylag/blob/master/pylag/mock.pyx) where `(u, v) = (1.0, 1.0)`.
Velocities are set in the instance method:
```Python
data_reader = MockVelocityEddyDiffusivityDataReader()
```

In [ ]:
from configparser import SafeConfigParser

import pylag.random as random
from pylag.numerics import StdNumMethod
from pylag.mock import MockTwoDNumMethod

warnings.filterwarnings('ignore')

In [ ]:
# Seed the random number generator
random.seed(10)

# Start/stop times
time_start = 0.0 # Start time (s)
time_end = 25.0  # End time (s)
time_step = 0.1  # Time step (s)

# PyLag reads many of its configuration parameters from a config which we create here
config = SafeConfigParser()

# Set the coordinate system, which here is a simple cartesian coordinate system
config.add_section('SIMULATION')
config.set('SIMULATION', 'coordinate_system', 'cartesian')

# Parameters controlling the type of numerical integration scheme used
config.add_section("NUMERICS")
config.set("NUMERICS", "num_method", "standard")
config.set("NUMERICS", "iterative_method", "AdvDiff_Milstein_3D")
config.set("NUMERICS", "time_step_diff", str(time_step))

# Parameters controlling particle vertical motion, which we set to default values
config.set("SIMULATION", "depth_restoring", "False")
config.set("SIMULATION", "fixed_depth", "0.0")

# 2024.03.26 nimura
# Activate ocean currents.
config.add_section("OCEAN_DATA")
config.set("OCEAN_DATA", "name", "dummy")

# The data reader
# Velocities are set.
data_reader = MockVelocityEddyDiffusivityDataReader()

# Numerical method for doing the actual integration
num_method_wrapper = MockTwoDNumMethod(config)

# Time points at which to compute particle positions
time = np.arange(time_start, time_end + time_step, time_step)

# Total number of time points at which to calcualte particle positions
n_times = len(time)

# The number of particles
n_particles = 1000

# Temporary arrays in which to store particle positions. All particles start at the origin.
x_positions = [0.0]*n_particles
y_positions = [0.0]*n_particles

# Arrays in which to save particle x and y positions
particle_x_positions = np.empty((n_times, n_particles), dtype=float)
particle_y_positions = np.empty((n_times, n_particles), dtype=float)

# Time integration
# ----------------

for t_idx, t in enumerate(time):
    # Save x and y positions for the last time point
    particle_x_positions[t_idx, :] = x_positions[:]
    particle_y_positions[t_idx, :] = y_positions[:]

    # Compute new x and y positions
    x_positions[:], y_positions[:] = num_method_wrapper.step(data_reader, t, x_positions, y_positions)

# Visualizing the result

# Trim arrays to match analytic solution
particle_x_positions = particle_x_positions[1:]
particle_y_positions = particle_y_positions[1:]

# Set up the plot, as before
fig, ax = create_figure(figure_size=(16., 16.), font_size=font_size)
ax.set_xlim((x_grid_edges.min(), x_grid_edges.max()))
ax.set_ylim((y_grid_edges.min(), y_grid_edges.max()))
ax.set_xlabel('x', fontsize=font_size)
ax.set_ylabel('y', fontsize=font_size)
ax.set_aspect('equal')

# Animate particle trajectories
scatter = ax.scatter([], [], c='r', s=20, zorder=2)

def animate(i):
    data = np.array([particle_x_positions[i], particle_y_positions[i]])
    scatter.set_offsets(data.transpose())
    return (scatter,)

# Prevent the basic figure from being plotted too
plt.close(fig)

anim = animation.FuncAnimation(fig, animate, frames=n_times-1, interval=40, blit=True)
anim.save("test02.gif")

HTML(anim.to_html5_video())

In [ ]:
from scipy import stats

# Define the grid
X, Y = np.meshgrid(x_grid, y_grid)
XY = np.vstack([X.ravel(), Y.ravel()])

# Compute the particle density field
density = np.empty((n_times-1, n_x_points, n_y_points), dtype=float)
for i in range(n_times-1):
    values = np.vstack([particle_x_positions[i, :], particle_y_positions[i, :]])
    kernel = stats.gaussian_kde(values)
    density[i, :, :] = np.reshape(kernel(XY).T, X.shape)

# Convert this into a concentration field
conc = density * data_reader.M

# Animate the result
# ------------------

font_size = 15
fig, ax = create_figure(figure_size=(16., 16.), font_size=font_size)
ax.set_xlim((x_grid_edges.min(), x_grid_edges.max()))
ax.set_ylim((y_grid_edges.min(), y_grid_edges.max()))
ax.set_xlabel('x', fontsize=font_size)
ax.set_ylabel('y', fontsize=font_size)
ax.set_aspect('equal')

# Set colour map and limits
cmap = colourmap('DYE')
vmin = 0.0
vmax = 10.0

# Set up the plot
pcolormesh= ax.pcolormesh(x_grid_edges, y_grid_edges, conc[0,:,:], vmin=vmin, vmax=vmax, cmap=cmap)
pos = ax.get_position().get_points()
cax = fig.add_axes([pos[1,0]+0.02, pos[0,1], 0.02, pos[1,1] - pos[0,1]])
cbar = fig.colorbar(pcolormesh, cax=cax)
cbar.ax.tick_params(labelsize=font_size)
cbar.set_label('C (kg m$\mathrm{^{-3}}$)', fontsize=font_size)

def animate(i):
    pcolormesh.set_array(conc[i,:,:].ravel())
    return (pcolormesh,)

# Prevent the basic figure from being plotted too
plt.close(fig)

anim = animation.FuncAnimation(fig, animate, frames=n_times-1, interval=40, blit=True)
anim.save("test03.gif")

HTML(anim.to_html5_video())